In [0]:
%pip install "numpy<2.0" faiss-cpu==1.8.0 --quiet

# COMMAND ----------
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 05 — RAG: Build FAISS Vector Index (v12 — Full Schema-Compatible Rebuild)
# MAGIC
# MAGIC **Rebuilt against verified schemas:**
# MAGIC - `gold_idp_enriched`        — 190 cols, IDP-enriched arrays, citations, readiness flags
# MAGIC - `gold_facilities_enriched_v12` — 179 cols, parsed arrays, quality/risk/desert/graph scores
# MAGIC - `gold_regional_summary_v12`    — 17 rows, one per Ghana region
# MAGIC
# MAGIC **Architecture:**
# MAGIC - Layer 1 : Canonical schema adapter (normalises all 3 sources into one record shape)
# MAGIC - Layer 2 : Document factories — facility doc, region doc (multi-view, quality-aware)
# MAGIC - Layer 3 : Metadata enrichment — quality, readiness, desert, risk, geo, anomaly, citations
# MAGIC - Layer 4 : BGE-large-en embeddings + hash-based cache + FAISS IndexFlatIP (cosine)
# MAGIC - Layer 5 : Intent-aware retrieval API (facility / service / region / planning / anomaly)
# MAGIC - Layer 6 : Map export + planning outputs + citation-backed synthesis

# COMMAND ----------
# MAGIC %pip install "numpy<2.0" faiss-cpu==1.8.0 --quiet

# COMMAND ----------
# dbutils.library.restartPython()

# COMMAND ----------
# MAGIC %md ## 0. Imports & Configuration

# COMMAND ----------

import json
import os
import pickle
import hashlib
import time
import re
from typing import List, Dict, Optional, Any, Tuple
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import requests
import faiss
import mlflow

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()
print(f"FAISS   : {faiss.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Run at  : {datetime.now(timezone.utc).isoformat()}")

# COMMAND ----------

class RAGConfig:
    # ── Databricks env ────────────────────────────────────────────────────────
    DATABRICKS_HOST  = spark.conf.get("spark.databricks.workspaceUrl", "").rstrip("/")
     # ✅ FIX: Use notebook authentication context (auto-rotates, secure)
    try:
        DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception as e:
        raise RuntimeError(f"❌ Failed to retrieve authentication token: {e}")

    # ── Models ────────────────────────────────────────────────────────────────
    EMBED_MODEL    = "databricks-bge-large-en"
    EMBED_ENDPOINT = f"https://{DATABRICKS_HOST}/serving-endpoints/databricks-bge-large-en/invocations"
    EMBED_DIM      = 1024

    LLM_MODEL    = "databricks-meta-llama-3-3-70b-instruct"
    LLM_ENDPOINT = f"https://{DATABRICKS_HOST}/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations"

    # ── Source tables ─────────────────────────────────────────────────────────
    IDP_TABLE      = "virtue_foundation.ghana.gold_idp_enriched"
    FACILITIES_TABLE = "virtue_foundation.ghana.gold_facilities_enriched"
    REGIONAL_TABLE = "virtue_foundation.ghana.gold_regional_summary"

    # ── Persistence paths ─────────────────────────────────────────────────────
    VOLUME_PATH      = "/Volumes/virtue_foundation/ghana"
    INDEX_PATH       = f"{VOLUME_PATH}/rag/faiss_index.bin"
    META_PATH        = f"{VOLUME_PATH}/rag/faiss_metadata.json"
    EMBED_CACHE_PATH = f"{VOLUME_PATH}/rag/embedding_cache.pkl"

    LOCAL_DIR        = "/Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/rag"
    LOCAL_INDEX_PATH = f"{LOCAL_DIR}/faiss_index.bin"
    LOCAL_META_PATH  = f"{LOCAL_DIR}/faiss_metadata.json"
    LOCAL_CACHE_PATH = f"{LOCAL_DIR}/embedding_cache.pkl"

    MLFLOW_EXP = "/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon"

    BATCH_SIZE       = 8
    RATE_LIMIT_DELAY = 1.0

    # ── WHO reference thresholds ──────────────────────────────────────────────
    LOW_DOCTOR_THRESHOLD   = 3
    LOW_CAPACITY_THRESHOLD = 10

    # ── Minimum quality to index ──────────────────────────────────────────────
    MIN_DOC_CHARS    = 30
    MIN_COMPLETENESS = 0.0   # include all rows; filter at query time

    # ── Schema version tag ───────────────────────────────────────────────────
    SCHEMA_VERSION = "v12"


cfg = RAGConfig()
# Add this test cell after RAGConfig definition
print(f"✅ Databricks Host: {cfg.DATABRICKS_HOST}")
print(f"✅ Token available: {bool(cfg.DATABRICKS_TOKEN)}")
print(f"✅ Embedding endpoint: {cfg.EMBED_ENDPOINT}")

# Test a single embedding
try:
    test_vec = embed_query("test query")
    print(f"✅ Embedding endpoint working! Vector shape: {test_vec.shape}")
except Exception as e:
    print(f"❌ Embedding test failed: {e}")
Path(cfg.LOCAL_DIR).mkdir(parents=True, exist_ok=True)
print(f"IDP table      : {cfg.IDP_TABLE}")
print(f"Facilities tbl : {cfg.FACILITIES_TABLE}")
print(f"Regional tbl   : {cfg.REGIONAL_TABLE}")
print(f"Local RAG dir  : {cfg.LOCAL_DIR}")

# COMMAND ----------
# MAGIC %md ## 1. Core Utility Functions

# COMMAND ----------

# ── ensure_list: handles every column encoding we encounter ──────────────────
# IDP enriched cols (procedure_enriched etc.) are JSON-encoded strings in Delta.
# Parsed array cols (procedure_parsed etc.) come as Python lists after toPandas().
# Regional summary list cols are also JSON strings.

def ensure_list(x) -> List[str]:
    """
    Convert any column value to a clean Python list of strings.
    Handles: Python list, JSON string, numpy ndarray, pandas/numpy NaN, None.
    """
    if x is None:
        return []
    # float NaN (pandas NaN is float)
    if isinstance(x, float) and (x != x):
        return []
    # numpy NaN / scalar
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray):
            return [str(v) for v in x.tolist()
                    if v is not None and str(v).strip() not in ("", "null", "nan", "None")]
        if isinstance(x, (_np.bool_, _np.integer, _np.floating)):
            return []
    except ImportError:
        pass
    if isinstance(x, list):
        return [str(v) for v in x
                if v is not None and str(v).strip() not in ("", "null", "nan", "None")]
    if isinstance(x, str):
        s = x.strip()
        if s in ("", "null", "[]", "nan", "None", "NaN"):
            return []
        # Try JSON
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(v) for v in parsed
                        if v is not None and str(v).strip() not in ("", "null", "None")]
            if parsed is None:
                return []
            return [str(parsed)]
        except (json.JSONDecodeError, ValueError):
            pass
        # Comma-separated plain text fallback
        if "," in s:
            return [v.strip().strip('"').strip("'")
                    for v in s.split(",") if v.strip()]
        return [s]
    return []


def safe_float(v, default: float = 0.0) -> float:
    try:
        if v is None:
            return default
        s = str(v).strip()
        if s in ("nan", "None", "null", "", "NaN"):
            return default
        return float(s)
    except (ValueError, TypeError):
        return default


def safe_int(v, default: int = 0) -> int:
    try:
        if v is None:
            return default
        s = str(v).strip()
        if s in ("nan", "None", "null", "", "NaN"):
            return default
        return int(float(s))
    except (ValueError, TypeError):
        return default


def safe_str(v, default: str = "") -> str:
    if v is None:
        return default
    s = str(v).strip()
    if s in ("None", "nan", "null", "", "NaN"):
        return default
    return s


def safe_bool(v, default: bool = False) -> bool:
    if v is None:
        return default
    if isinstance(v, bool):
        return v
    try:
        import numpy as _np
        if isinstance(v, _np.bool_):
            return bool(v)
    except ImportError:
        pass
    s = str(v).strip().lower()
    if s in ("true", "1", "yes", "t"):
        return True
    if s in ("false", "0", "no", "f", "nan", "none", "null", ""):
        return False
    return default


# ── Clinical junk filter (address/contact/meta noise) ─────────────────────────
_JUNK_PATTERNS = [
    r"(?i)^located\s+(at|in)",
    r"(?i)^p\.?\s*o\.?\s*box\s+",
    r"(?i)GPS\s+address",
    r"(?i)official\s+website|website\s*:",
    r"(?i)facebook\s+page|instagram|twitter",
    r"(?i)^always\s+open$",
    r"(?i)^phone\s*(number|contact)?:",
    r"(?i)telephone|email[:\s]",
    r"(?i)^fax[:\s]",
    r"(?i)^\+?\d{7,}",
    r"http[s]?://",
    r"(?i)opening\s+hours?|business\s+hours?",
    r"(?i)^address\s*:",
    r"(?i)^type\s*:\s*(primary|secondary|tertiary|district|regional|community)",
    r"(?i)^ownership\s*:\s*(government|private|public|mission|faith)",
    r"(?i)nhis\s*accredited",
]


def _is_clean_clinical_item(text: str) -> bool:
    if not text or len(text.strip()) < 8:
        return False
    return not any(re.search(p, text) for p in _JUNK_PATTERNS)


# ── Clinical hint pattern (for filtering description sentences) ───────────────
_CLINICAL_HINTS = re.compile(
    r"(?i)\b("
    r"surgery|surgical|theatre|operating|operation|procedure|treatment|"
    r"ICU|intensive care|emergency|casualty|trauma|obstetric|maternity|"
    r"dialysis|radiology|imaging|x.?ray|CT scan|MRI|ultrasound|"
    r"laboratory|lab\b|diagnostic|dental|ophthalmolog|pediatric|paediatric|"
    r"neonatal|NICU|burn unit|ward|inpatient|outpatient|admission|"
    r"cancer|oncolog|cardiac|heart|neurol|psychiatr|mental health|"
    r"HIV|AIDS|tuberculosis|TB|malaria|infectious|immunis|vaccine|"
    r"antenatal|postnatal|delivery|blood bank|pharmacy"
    r")\b"
)


# ── Sanity-check ensure_list ──────────────────────────────────────────────────
_tests = [
    ('["a","b"]', 2), ('[]', 0), (None, 0), ([], 0),
    (["a", "b"], 2), ('["internalMedicine"]', 1),
    ('null', 0), ("single", 1), (float('nan'), 0),
]
for inp, expected in _tests:
    result = ensure_list(inp)
    status = "✅" if len(result) == expected else f"❌ got {len(result)}"
    print(f"  {status}  ensure_list({str(inp)[:30]!r}) → {result}")

# COMMAND ----------
# MAGIC %md ## 2. Canonical Schema Adapter
# MAGIC
# MAGIC Normalises rows from gold_idp_enriched (190 cols), gold_facilities_enriched_v12 (179 cols)
# MAGIC and gold_regional_summary_v12 (17 rows) into one internal canonical dict.

# COMMAND ----------

def _get(row: pd.Series, *keys, default="") -> Any:
    """Try each key in order; return first non-null, non-empty hit."""
    for k in keys:
        v = row.get(k)
        if v is None:
            continue
        if isinstance(v, float) and v != v:
            continue
        s = str(v).strip()
        if s not in ("None", "nan", "null", "", "NaN"):
            return v
    return default


def canonical_facility_row(row: pd.Series, source_table: str) -> Dict[str, Any]:
    """
    Convert one row from either IDP or facilities table into a single
    canonical dict with stable keys used throughout the pipeline.
    All aliasing / version differences live here and nowhere else.
    """
    # ── Identity ──────────────────────────────────────────────────────────────
    rec: Dict[str, Any] = {
        "doc_type":       "facility",
        "schema_version": cfg.SCHEMA_VERSION,
        "source_table":   source_table,

        "unique_id":      safe_str(_get(row, "unique_id")),
        "name":           safe_str(_get(row, "name")),
        "pk_unique_id":   safe_str(_get(row, "pk_unique_id")),

        # Facility classification
        "facility_type":      safe_str(_get(row, "facility_type_clean", "facilityTypeId")),
        "operator_type":      safe_str(_get(row, "operatorTypeId", "operator_type_raw")),
        "organization_type":  safe_str(_get(row, "organization_type_clean", "organization_type")),
        "organization_category": safe_str(_get(row, "organization_category")),
        "ownership_model":    safe_str(_get(row, "ownership_model")),
        "facility_tier_label":   safe_str(_get(row, "facility_tier_label")),
        "facility_complexity":   safe_str(_get(row, "facility_complexity_level")),

        # Boolean type flags
        "is_hospital":     safe_bool(_get(row, "is_hospital")),
        "is_clinic":       safe_bool(_get(row, "is_clinic")),
        "is_ngo":          safe_bool(_get(row, "is_ngo")),
        "is_public":       safe_bool(_get(row, "is_public")),
        "is_private":      safe_bool(_get(row, "is_private")),
        "is_faith_based":  safe_bool(_get(row, "is_faith_based")),
        "is_government":   safe_bool(_get(row, "is_government")),
        "is_teaching_hospital":  safe_bool(_get(row, "is_teaching_hospital")),
        "is_referral_center":    safe_bool(_get(row, "is_referral_center")),
        "is_specialist_hospital":safe_bool(_get(row, "is_specialist_hospital")),

        # Location — normalise casing differences across schema versions
        "region":    safe_str(_get(row, "region_normalised", "address_stateOrRegion",
                                        "address_stateorregion")),
        "city":      safe_str(_get(row, "city_clean", "address_city")),
        "address":   safe_str(_get(row, "address_line1")),
        "latitude":  safe_float(_get(row, "latitude"), 7.9465),
        "longitude": safe_float(_get(row, "longitude"), -1.0232),

        # Geo quality
        "geo_quality_score":       safe_float(_get(row, "geo_quality_score"), 0.5),
        "geo_contradiction_flag":  safe_bool(_get(row, "geo_contradiction_flag")),
        "geo_region_mismatch":     safe_bool(_get(row, "geo_region_mismatch")),

        # Contact
        "email":    safe_str(_get(row, "email")),
        "website":  safe_str(_get(row, "officialWebsite", "officialwebsite")),
        "source_url": safe_str(_get(row, "source_url")),

        # Clinical arrays — prefer IDP-enriched; fall back to parsed; then raw string
        "specialties": (
            ensure_list(_get(row, "specialties_enriched"))
            or ensure_list(_get(row, "specialties_parsed"))
            or ensure_list(_get(row, "specialties"))
        ),
        "procedures": [v for v in (
            ensure_list(_get(row, "procedure_enriched"))
            or ensure_list(_get(row, "procedure_parsed"))
            or ensure_list(_get(row, "procedure"))
        ) if _is_clean_clinical_item(v)],
        "equipment": [v for v in (
            ensure_list(_get(row, "equipment_enriched"))
            or ensure_list(_get(row, "equipment_parsed"))
            or ensure_list(_get(row, "equipment"))
        ) if _is_clean_clinical_item(v)],
        "capabilities": [v for v in (
            ensure_list(_get(row, "capability_enriched"))
            or ensure_list(_get(row, "capability_parsed"))
            or ensure_list(_get(row, "capability"))
        ) if _is_clean_clinical_item(v)],

        # Clinical specialty boolean flags
        "has_emergency_medicine": safe_bool(_get(row, "has_emergency_medicine")),
        "has_surgery":            safe_bool(_get(row, "has_surgery")),
        "has_obstetrics":         safe_bool(_get(row, "has_obstetrics")),
        "has_pediatrics":         safe_bool(_get(row, "has_pediatrics")),
        "has_icu":                safe_bool(_get(row, "has_icu")),
        "has_radiology":          safe_bool(_get(row, "has_radiology")),
        "has_infectious_disease": safe_bool(_get(row, "has_infectious_disease")),
        "has_mental_health":      safe_bool(_get(row, "has_mental_health")),

        # Numeric
        "doctors":          safe_int(_get(row, "number_doctors_int", "numberDoctors")),
        "capacity":         safe_int(_get(row, "capacity_int", "capacity")),
        "year_established": safe_int(_get(row, "year_established_int", "yearestablished")),

        # Volunteer acceptance — bool in schema but sometimes stored as string
        "accepts_volunteers": (
            safe_bool(_get(row, "accepts_volunteers_bool"))
            or str(_get(row, "acceptsvolunteers", "")).strip().lower() in ("true", "1", "yes")
        ),

        # Quality / readiness signals
        "is_rag_ready":        safe_bool(_get(row, "is_rag_ready"), True),
        "is_search_ready":     safe_bool(_get(row, "is_search_ready"), True),
        "is_planning_ready":   safe_bool(_get(row, "is_planning_ready"), True),
        "is_clinical_ready":   safe_bool(_get(row, "is_clinical_ready"), True),
        "data_completeness_score": safe_float(_get(row, "data_completeness_score"), 0.0),
        "rag_quality_score":       safe_float(_get(row, "rag_quality_score"), 0.0),
        "clinical_completeness":   safe_float(_get(row, "clinical_completeness"), 0.0),
        "location_completeness":   safe_float(_get(row, "location_completeness"), 0.0),
        "contact_completeness":    safe_float(_get(row, "contact_completeness"), 0.0),
        "evidence_weight":         safe_float(_get(row, "evidence_weight"), 0.5),
        "source_trust":            safe_str(_get(row, "source_trust")),
        "row_quality_flags":       ensure_list(_get(row, "row_quality_flags")),
        "quality_flag_count":      safe_int(_get(row, "quality_flag_count")),

        # Risk scores
        "quality_risk_score":    safe_float(_get(row, "quality_risk_score"), 0.0),
        "clinical_risk_score":   safe_float(_get(row, "clinical_risk_score"), 0.0),
        "operational_risk_score":safe_float(_get(row, "operational_risk_score"), 0.0),
        "integrity_risk_score":  safe_float(_get(row, "integrity_risk_score"), 0.0),
        "clinical_complexity_score": safe_float(_get(row, "clinical_complexity_score"), 0.0),
        "emergency_readiness_score": safe_float(_get(row, "emergency_readiness_score"), 0.0),
        "critical_care_score":       safe_float(_get(row, "critical_care_score"), 0.0),
        "capability_graph_nodes":    ensure_list(_get(row, "capability_graph_nodes")),
        "capability_graph_edges":    ensure_list(_get(row, "capability_graph_edges")),
        "capability_dependency_gaps": ensure_list(_get(row, "capability_dependency_gaps")),
        "capability_graph_summary":  safe_str(_get(row, "capability_graph_summary")),
        "service_richness_score":    safe_float(_get(row, "service_richness_score"), 0.0),
        "infrastructure_completeness_score": safe_float(_get(row, "infrastructure_completeness_score"), 0.0),
        "referral_complexity_score": safe_float(_get(row, "referral_complexity_score"), 0.0),
        "healthcare_maturity_score": safe_float(_get(row, "healthcare_maturity_score"), 0.0),
        "capability_graph_version":  safe_str(_get(row, "capability_graph_version")),

        # Anomaly flags
        "stat_anomaly_capability_inflation": safe_bool(_get(row, "stat_anomaly_capability_inflation")),
        "stat_anomaly_hospital_no_doctors":  safe_bool(_get(row, "stat_anomaly_hospital_no_doctors")),
        "stat_anomaly_clinic_claims_icu":    safe_bool(_get(row, "stat_anomaly_clinic_claims_icu")),
        "stat_anomaly_ghost_facility":       safe_bool(_get(row, "stat_anomaly_ghost_facility")),
        "stat_anomaly_specialty_mismatch":   safe_bool(_get(row, "stat_anomaly_specialty_mismatch")),
        "stat_anomaly_procedure_breadth":    safe_bool(_get(row, "stat_anomaly_procedure_breadth")),
        "total_stat_anomalies":              safe_int(_get(row, "total_stat_anomalies")),
        "ghost_probability_score":           safe_float(_get(row, "ghost_probability_score"), 0.0),
        "capability_is_valid":               safe_bool(_get(row, "capability_is_valid"), True),
        "capability_anomalies":              ensure_list(_get(row, "capability_anomalies")),
        "capability_confidence":             safe_float(_get(row, "capability_confidence"), 1.0),
        "has_anomaly":                       (not safe_bool(_get(row, "capability_is_valid"), True))
                                             or (safe_int(_get(row, "total_stat_anomalies")) > 0),

        # Medical desert
        "medical_desert_score": safe_float(_get(row, "medical_desert_score"), 0.5),
        "desert_label":         safe_str(_get(row, "desert_label"), "Unknown"),

        # IDP provenance
        "idp_run_id":     safe_str(_get(row, "idp_run_id")),
        "citation_row_id":safe_str(_get(row, "citation_row_id")),
        "idp_citations":  [c for c in ensure_list(_get(row, "idp_citations", "citations"))
                           if len(c) > 15 and _is_clean_clinical_item(c)],

        # Pre-built doc_text (from pipeline; use if quality-ready)
        "doc_text":        safe_str(_get(row, "doc_text")),
        "doc_text_length": safe_int(_get(row, "doc_text_length")),

        # Descriptions
        "description":           safe_str(_get(row, "description")),
        "organizationdescription": safe_str(_get(row, "organizationdescription")),
        "missionstatement":        safe_str(_get(row, "missionstatement")),
        "ngo_serves_ghana":        safe_bool(_get(row, "ngo_serves_ghana")),
    }
    return rec


def canonical_region_row(row: pd.Series) -> Dict[str, Any]:
    """
    Convert one row from gold_regional_summary_v12 into canonical regional doc.
    """
    region = safe_str(_get(row, "region_normalised"), "Unknown")
    mds    = safe_float(_get(row, "medical_desert_score"), 0.5)
    dlabel = safe_str(_get(row, "desert_label"), "Unknown")

    rec: Dict[str, Any] = {
        "doc_type":       "region",
        "schema_version": cfg.SCHEMA_VERSION,
        "source_table":   cfg.REGIONAL_TABLE,

        "unique_id": f"region_{region.lower().replace(' ', '_')}",
        "name":      region,
        "region":    region,
        "city":      region,

        "latitude":  safe_float(_get(row, "region_centroid_lat"), 7.9465),
        "longitude": safe_float(_get(row, "region_centroid_lon"), -1.0232),

        # Counts
        "total_facilities":       safe_int(_get(row, "total_facilities")),
        "hospital_count":         safe_int(_get(row, "hospital_count")),
        "clinic_count":           safe_int(_get(row, "clinic_count")),
        "ngo_count":              safe_int(_get(row, "ngo_count")),
        "government_facilities":  safe_int(_get(row, "government_facilities")),
        "teaching_hospital_count":safe_int(_get(row, "teaching_hospital_count")),
        "referral_center_count":  safe_int(_get(row, "referral_center_count")),
        "volunteer_facilities":   safe_int(_get(row, "volunteer_facilities")),

        # Workforce / capacity
        "total_doctors": safe_int(_get(row, "total_doctors")),
        "avg_doctors":   safe_float(_get(row, "avg_doctors")),
        "total_beds":    safe_int(_get(row, "total_beds")),
        "avg_bed_capacity": safe_float(_get(row, "avg_bed_capacity")),

        # Per-capita
        "doctors_per_100k":           safe_float(_get(row, "doctors_per_100k")),
        "beds_per_100k":              safe_float(_get(row, "beds_per_100k")),
        "facilities_per_100k":        safe_float(_get(row, "facilities_per_100k")),
        "hospitals_per_100k":         safe_float(_get(row, "hospitals_per_100k")),
        "icu_facilities_per_100k":    safe_float(_get(row, "icu_facilities_per_100k")),
        "surgery_facilities_per_100k":safe_float(_get(row, "surgery_facilities_per_100k")),
        "maternity_facilities_per_100k": safe_float(_get(row, "maternity_facilities_per_100k")),

        # Specialty coverage
        "emergency_medicine_facilities": safe_int(_get(row, "emergency_medicine_facilities")),
        "obstetrics_facilities":         safe_int(_get(row, "obstetrics_facilities")),
        "surgery_facilities":            safe_int(_get(row, "surgery_facilities")),
        "pediatrics_facilities":         safe_int(_get(row, "pediatrics_facilities")),
        "icu_facilities":                safe_int(_get(row, "icu_facilities")),
        "infectious_disease_facilities": safe_int(_get(row, "infectious_disease_facilities")),
        "radiology_facilities":          safe_int(_get(row, "radiology_facilities")),
        "mental_health_facilities":      safe_int(_get(row, "mental_health_facilities")),

        # Gap scores
        "maternity_gap_score":       safe_float(_get(row, "maternity_gap_score")),
        "emergency_gap_score":       safe_float(_get(row, "emergency_gap_score")),
        "icu_gap_score":             safe_float(_get(row, "icu_gap_score")),
        "surgical_access_gap_score": safe_float(_get(row, "surgical_access_gap_score")),

        # Quality / desert
        "medical_desert_score":   mds,
        "desert_label":           dlabel,
        "avg_quality_risk":       safe_float(_get(row, "avg_quality_risk")),
        "avg_completeness":       safe_float(_get(row, "avg_completeness")),
        "avg_emergency_readiness":safe_float(_get(row, "avg_emergency_readiness")),
        "avg_critical_care_score":safe_float(_get(row, "avg_critical_care_score")),
        "avg_service_richness_score": safe_float(_get(row, "avg_service_richness_score")),
        "avg_infrastructure_completeness_score": safe_float(_get(row, "avg_infrastructure_completeness_score")),
        "avg_referral_complexity_score": safe_float(_get(row, "avg_referral_complexity_score")),
        "avg_healthcare_maturity_score": safe_float(_get(row, "avg_healthcare_maturity_score")),
        "total_region_anomalies": safe_int(_get(row, "total_region_anomalies")),
        "rag_ready_count":        safe_int(_get(row, "rag_ready_count")),
        "rag_ready_rate":         safe_float(_get(row, "rag_ready_rate")),

        # Specialty gaps
        "all_specialties":              ensure_list(_get(row, "all_specialties")),
        "missing_critical_specialties": ensure_list(_get(row, "missing_critical_specialties")),
        "critical_specialty_gap_count": safe_int(_get(row, "critical_specialty_gap_count")),
        "recommended_actions":          ensure_list(_get(row, "recommended_actions")),

        # Derived risk label
        "risk_level": (
            "HIGH"   if mds > 0.6 else
            "MEDIUM" if mds > 0.35 else
            "LOW"
        ),
    }
    return rec

# COMMAND ----------
# MAGIC %md ## 3. Document Factories

# COMMAND ----------

def build_facility_document(rec: Dict[str, Any]) -> str:
    """
    Build a dense, semantically-rich text document for one canonical facility record.

    Strategy:
    1. If `doc_text` is available and quality-ready → use it directly (pipeline-computed).
    2. Otherwise → compose from canonical fields (fallback builder).

    The text is optimised for cosine-similarity retrieval with BGE-large-en.
    """
    # ── Fast path: use pre-built doc_text when quality-ready ──────────────────
    dt = rec.get("doc_text", "")
    if (dt and len(dt) > cfg.MIN_DOC_CHARS
            and rec.get("is_rag_ready", True)):
        # Append enriched arrays not already in doc_text (IDP-only upgrade)
        extra_parts: List[str] = []

        for label, key in [("Procedures", "procedures"),
                           ("Equipment", "equipment"),
                           ("Capabilities", "capabilities"),
                           ("Specialties", "specialties")]:
            items = rec.get(key, [])
            if items:
                snippet = f"{label}: {', '.join(items[:8])}"
                if snippet.lower()[:20] not in dt.lower()[:2000]:
                    extra_parts.append(snippet)

        # Volunteer keyword (boosts volunteer filter recall)
        if rec.get("accepts_volunteers"):
            extra_parts.append("Accepts volunteer doctors and healthcare workers")

        # Medical desert label
        dlabel = rec.get("desert_label", "")
        mds    = rec.get("medical_desert_score", 0.5)
        if dlabel and dlabel not in dt:
            extra_parts.append(f"Medical Desert Status: {dlabel} (score={mds:.3f})")

        dep_gaps = rec.get("capability_dependency_gaps", [])
        if dep_gaps:
            extra_parts.append(f"Capability dependency gaps: {', '.join(dep_gaps[:8])}")
        maturity = rec.get("healthcare_maturity_score", 0.0)
        infra = rec.get("infrastructure_completeness_score", 0.0)
        if maturity or infra:
            extra_parts.append(
                f"Healthcare maturity score: {maturity:.3f}; infrastructure completeness: {infra:.3f}"
            )

        # IDP citations (up to 2)
        for c in rec.get("idp_citations", [])[:2]:
            if c and len(c) > 20 and c[:50] not in dt:
                extra_parts.append(f"Source evidence: {c[:200]}")

        if extra_parts:
            return dt.rstrip(".") + ". " + ". ".join(extra_parts)
        return dt

    # ── Fallback builder: compose from canonical fields ───────────────────────
    parts: List[str] = []

    name = rec.get("name", "")
    if name:
        parts.append(f"Facility: {name}")

    type_parts = [t for t in [
        rec.get("facility_type", ""),
        rec.get("operator_type", ""),
        rec.get("organization_type", ""),
        rec.get("facility_tier_label", ""),
    ] if t]
    if type_parts:
        parts.append(f"Type: {', '.join(type_parts)}")

    loc_parts = [v for v in [rec.get("city", ""), rec.get("region", ""), "Ghana"] if v]
    parts.append(f"Location: {', '.join(loc_parts)}")

    mds    = rec.get("medical_desert_score", 0.5)
    dlabel = rec.get("desert_label", "Unknown")
    parts.append(f"Medical Desert Status: {dlabel} (score={mds:.3f})")

    if rec.get("specialties"):
        parts.append(f"Medical specialties: {', '.join(rec['specialties'])}")

    for label, key, max_items in [
        ("Procedures",   "procedures",   10),
        ("Equipment",    "equipment",    8),
        ("Capabilities", "capabilities", 8),
    ]:
        items = rec.get(key, [])
        if items:
            parts.append(f"{label}: {'. '.join(items[:max_items])}")

    # Boolean capability flags → readable phrases
    bool_flags: List[str] = []
    flag_map = {
        "has_emergency_medicine": "24-hour emergency medicine",
        "has_surgery":            "surgical services",
        "has_obstetrics":         "obstetrics and maternity care",
        "has_pediatrics":         "pediatric care",
        "has_icu":                "intensive care unit (ICU)",
        "has_radiology":          "radiology and imaging",
        "has_infectious_disease": "infectious disease treatment (HIV, TB, malaria)",
        "has_mental_health":      "mental health services",
    }
    for col, label in flag_map.items():
        if rec.get(col):
            bool_flags.append(label)
    if bool_flags:
        parts.append(f"Confirmed capabilities: {', '.join(bool_flags)}")

    dep_gaps = rec.get("capability_dependency_gaps", [])
    if dep_gaps:
        parts.append(f"Capability dependency gaps: {', '.join(dep_gaps[:10])}")
    if rec.get("healthcare_maturity_score", 0.0) or rec.get("infrastructure_completeness_score", 0.0):
        parts.append(
            "Capability graph scores: "
            f"maturity={rec.get('healthcare_maturity_score', 0.0):.3f}, "
            f"infrastructure_completeness={rec.get('infrastructure_completeness_score', 0.0):.3f}, "
            f"service_richness={rec.get('service_richness_score', 0.0):.3f}, "
            f"referral_complexity={rec.get('referral_complexity_score', 0.0):.3f}"
        )

    if not rec.get("capability_is_valid", True):
        parts.append("WARNING: Capability claims flagged as potentially invalid by IDP validator")
    if rec.get("total_stat_anomalies", 0) > 0:
        parts.append(f"Statistical anomaly flags: {rec['total_stat_anomalies']}")
    if rec.get("ghost_probability_score", 0.0) > 0.5:
        parts.append(f"Ghost facility probability: {rec['ghost_probability_score']:.2f}")

    # Clinical sentences from description / mission
    for field in ["description", "organizationdescription"]:
        v = rec.get(field, "")
        if v and len(v) > 40:
            sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", v[:1200])
                     if re.search(_CLINICAL_HINTS, s)]
            if sents:
                parts.append(f"Description: {' '.join(sents[:3])[:400]}")
                break
    mission = rec.get("missionstatement", "")
    if mission and len(mission) > 30:
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", mission[:800])
                 if re.search(_CLINICAL_HINTS, s)]
        if sents:
            parts.append(f"Mission: {' '.join(sents[:2])[:300]}")

    for field, label in [
        ("doctors",          "Doctors"),
        ("capacity",         "Beds"),
        ("year_established", "Established"),
    ]:
        v = rec.get(field, 0)
        if v and v > 0:
            parts.append(f"{label}: {v}")

    if rec.get("accepts_volunteers"):
        parts.append("Accepts volunteer doctors and healthcare workers")

    for c in rec.get("idp_citations", [])[:2]:
        if len(c) > 20:
            parts.append(f"Source evidence: {c[:200]}")

    return ". ".join(p.strip() for p in parts if p.strip())


def build_region_document(rec: Dict[str, Any]) -> str:
    """
    Build a text document for one Ghana region from canonical region record.
    Optimised for regional/planning/gap queries.
    """
    parts: List[str] = []
    region = rec.get("name", "Unknown")
    parts.append(f"Region: {region}, Ghana")

    mds    = rec.get("medical_desert_score", 0.5)
    dlabel = rec.get("desert_label", "Unknown")
    parts.append(f"Medical Desert Status: {dlabel} (score={mds:.3f})")

    # Counts
    parts.append(
        f"Healthcare infrastructure: {rec.get('total_facilities', 0)} total facilities, "
        f"{rec.get('hospital_count', 0)} hospitals, "
        f"{rec.get('clinic_count', 0)} clinics, "
        f"{rec.get('ngo_count', 0)} NGOs, "
        f"{rec.get('government_facilities', 0)} government facilities"
    )

    # Workforce
    parts.append(
        f"Workforce: {rec.get('total_doctors', 0)} doctors total "
        f"({rec.get('doctors_per_100k', 0):.2f} per 100k population), "
        f"{rec.get('total_beds', 0)} beds total "
        f"({rec.get('beds_per_100k', 0):.2f} per 100k)"
    )

    # Specialty coverage
    specialty_parts = []
    for label, key in [
        ("emergency medicine", "emergency_medicine_facilities"),
        ("surgery",            "surgery_facilities"),
        ("obstetrics",         "obstetrics_facilities"),
        ("pediatrics",         "pediatrics_facilities"),
        ("ICU",                "icu_facilities"),
        ("infectious disease", "infectious_disease_facilities"),
        ("radiology",          "radiology_facilities"),
        ("mental health",      "mental_health_facilities"),
    ]:
        count = rec.get(key, 0)
        if count and count > 0:
            specialty_parts.append(f"{count} with {label}")
    if specialty_parts:
        parts.append(f"Specialty coverage: {', '.join(specialty_parts)}")

    # Gap scores
    gap_parts = []
    for label, key in [
        ("emergency access gap", "emergency_gap_score"),
        ("ICU gap",              "icu_gap_score"),
        ("surgical access gap",  "surgical_access_gap_score"),
        ("maternity gap",        "maternity_gap_score"),
    ]:
        v = rec.get(key, 0.0)
        if v and v > 0.1:
            gap_parts.append(f"{label}={v:.2f}")
    if gap_parts:
        parts.append(f"Healthcare gaps: {', '.join(gap_parts)}")

    # Missing specialties
    missing = rec.get("missing_critical_specialties", [])
    if missing:
        parts.append(f"Missing critical specialties: {', '.join(missing[:6])}")

    # Recommended actions
    actions = rec.get("recommended_actions", [])
    if actions:
        parts.append(f"Recommended interventions: {'; '.join(actions[:3])}")

    # Quality
    parts.append(
        f"Data quality: avg completeness={rec.get('avg_completeness', 0):.2f}, "
        f"total anomalies={rec.get('total_region_anomalies', 0)}, "
        f"RAG-ready facilities={rec.get('rag_ready_count', 0)} "
        f"({rec.get('rag_ready_rate', 0):.0%})"
    )

    # Readiness
    parts.append(
        f"Readiness scores: avg emergency readiness={rec.get('avg_emergency_readiness', 0):.2f}, "
        f"avg critical care={rec.get('avg_critical_care_score', 0):.2f}"
    )

    return ". ".join(p.strip() for p in parts if p.strip())

# COMMAND ----------
# MAGIC %md ## 4. Metadata Builders

# COMMAND ----------

def build_facility_metadata(rec: Dict[str, Any]) -> Dict[str, Any]:
    """
    Build the metadata dict stored alongside each FAISS vector for a facility.
    Used for post-retrieval filtering, popup rendering, and planning system.
    Adds inferred gaps / needs / risk_level fields.
    """
    mds = rec.get("medical_desert_score", 0.5)
    risk = (
        "HIGH"   if mds > 0.6 else
        "MEDIUM" if mds > 0.35 else
        "LOW"
    )

    meta = {
        # Document identity
        "doc_type":       "facility",
        "schema_version": cfg.SCHEMA_VERSION,
        "source_table":   rec.get("source_table", ""),
        # Facility identity
        "unique_id":  rec.get("unique_id", ""),
        "name":       rec.get("name", ""),
        # Classification
        "facility_type":      rec.get("facility_type", "unknown"),
        "operator_type":      rec.get("operator_type", ""),
        "organization_type":  rec.get("organization_type", ""),
        "organization_category": rec.get("organization_category", ""),
        "facility_tier_label":   rec.get("facility_tier_label", ""),
        # Type flags
        "is_hospital":  rec.get("is_hospital", False),
        "is_clinic":    rec.get("is_clinic", False),
        "is_ngo":       rec.get("is_ngo", False),
        "is_public":    rec.get("is_public", False),
        "is_private":   rec.get("is_private", False),
        "is_faith_based": rec.get("is_faith_based", False),
        "is_government":  rec.get("is_government", False),
        "is_teaching_hospital":   rec.get("is_teaching_hospital", False),
        "is_referral_center":     rec.get("is_referral_center", False),
        "is_specialist_hospital": rec.get("is_specialist_hospital", False),
        # Location
        "region":    rec.get("region", "Unknown"),
        "city":      rec.get("city", "Unknown"),
        "address":   rec.get("address", ""),
        "latitude":  rec.get("latitude", 7.9465),
        "longitude": rec.get("longitude", -1.0232),
        # Clinical arrays
        "specialties":  rec.get("specialties", []),
        "procedures":   rec.get("procedures", []),
        "equipment":    rec.get("equipment", []),
        "capabilities": rec.get("capabilities", []),
        # Specialty boolean flags
        "has_emergency_medicine": rec.get("has_emergency_medicine", False),
        "has_surgery":            rec.get("has_surgery", False),
        "has_obstetrics":         rec.get("has_obstetrics", False),
        "has_pediatrics":         rec.get("has_pediatrics", False),
        "has_icu":                rec.get("has_icu", False),
        "has_radiology":          rec.get("has_radiology", False),
        "has_infectious_disease": rec.get("has_infectious_disease", False),
        "has_mental_health":      rec.get("has_mental_health", False),
        # Numeric
        "doctors":          rec.get("doctors", 0),
        "capacity":         rec.get("capacity", 0),
        "year_established": rec.get("year_established", 0),
        # Volunteer
        "accepts_volunteers": rec.get("accepts_volunteers", False),
        # Readiness
        "is_rag_ready":      rec.get("is_rag_ready", True),
        "is_search_ready":   rec.get("is_search_ready", True),
        "is_planning_ready": rec.get("is_planning_ready", True),
        "is_clinical_ready": rec.get("is_clinical_ready", True),
        # Quality
        "data_completeness_score": rec.get("data_completeness_score", 0.0),
        "rag_quality_score":       rec.get("rag_quality_score", 0.0),
        "clinical_completeness":   rec.get("clinical_completeness", 0.0),
        "location_completeness":   rec.get("location_completeness", 0.0),
        "evidence_weight":         rec.get("evidence_weight", 0.5),
        "source_trust":            rec.get("source_trust", ""),
        "row_quality_flags":       rec.get("row_quality_flags", []),
        "quality_flag_count":      rec.get("quality_flag_count", 0),
        # Risk scores
        "quality_risk_score":     rec.get("quality_risk_score", 0.0),
        "clinical_risk_score":    rec.get("clinical_risk_score", 0.0),
        "operational_risk_score": rec.get("operational_risk_score", 0.0),
        "integrity_risk_score":   rec.get("integrity_risk_score", 0.0),
        "emergency_readiness_score": rec.get("emergency_readiness_score", 0.0),
        "critical_care_score":       rec.get("critical_care_score", 0.0),
        "capability_dependency_gaps": rec.get("capability_dependency_gaps", []),
        "service_richness_score":    rec.get("service_richness_score", 0.0),
        "infrastructure_completeness_score": rec.get("infrastructure_completeness_score", 0.0),
        "referral_complexity_score": rec.get("referral_complexity_score", 0.0),
        "healthcare_maturity_score": rec.get("healthcare_maturity_score", 0.0),
        "capability_graph_version":  rec.get("capability_graph_version", ""),
        # Anomaly
        "total_stat_anomalies": rec.get("total_stat_anomalies", 0),
        "has_anomaly":          rec.get("has_anomaly", False),
        "capability_is_valid":  rec.get("capability_is_valid", True),
        "capability_anomalies": rec.get("capability_anomalies", []),
        "capability_confidence":rec.get("capability_confidence", 1.0),
        "ghost_probability_score": rec.get("ghost_probability_score", 0.0),
        # Medical desert
        "medical_desert_score": mds,
        "desert_label":         rec.get("desert_label", "Unknown"),
        "risk_level":           risk,
        # Provenance
        "idp_run_id":      rec.get("idp_run_id", ""),
        "citation_row_id": rec.get("citation_row_id", ""),
        "idp_citations":   rec.get("idp_citations", []),
        # Contact
        "email":      rec.get("email", ""),
        "website":    rec.get("website", ""),
        "source_url": rec.get("source_url", ""),
        "ngo_serves_ghana": rec.get("ngo_serves_ghana", False),
    }

    # ── Planning system fields: infer gaps & needs ─────────────────────────────
    gaps:  List[str] = []
    needs: List[str] = []

    if not meta["has_emergency_medicine"]:
        gaps.append("No emergency care available")
        needs.append("Emergency medicine doctors required")
    if not meta["has_surgery"]:
        gaps.append("No surgical capability")
        needs.append("Basic surgical infrastructure needed")
    if not meta["has_obstetrics"]:
        gaps.append("No obstetric/maternity services")
        needs.append("Obstetrics support required")
    if not meta["has_icu"]:
        gaps.append("No ICU / critical care unit")
        needs.append("ICU setup and ventilators required")
    if not meta["has_infectious_disease"]:
        gaps.append("No infectious disease (HIV/TB/malaria) capability")
        needs.append("Infectious disease programme needed")
    if meta["doctors"] < cfg.LOW_DOCTOR_THRESHOLD and meta["doctors"] >= 0:
        gaps.append(f"Severely understaffed (only {meta['doctors']} doctors)")
        needs.append("Increase doctor count urgently")
    if meta["capacity"] < cfg.LOW_CAPACITY_THRESHOLD and meta["capacity"] >= 0:
        gaps.append(f"Very low bed capacity ({meta['capacity']} beds)")
        needs.append("Expand inpatient capacity")
    if meta["has_anomaly"]:
        gaps.append("Capability data anomaly detected — claims require verification")
        needs.append("Field audit of capability claims recommended")
    if meta["ghost_probability_score"] > 0.5:
        gaps.append(f"Possible ghost facility (probability={meta['ghost_probability_score']:.2f})")
        needs.append("Field verification of facility existence required")

    meta["medical_gaps"]  = gaps
    meta["medical_needs"] = needs

    return meta


def build_region_metadata(rec: Dict[str, Any]) -> Dict[str, Any]:
    """
    Build the metadata dict stored alongside each FAISS vector for a region.
    """
    mds = rec.get("medical_desert_score", 0.5)
    meta = {
        "doc_type":       "region",
        "schema_version": cfg.SCHEMA_VERSION,
        "source_table":   cfg.REGIONAL_TABLE,
        "unique_id":      rec.get("unique_id", ""),
        "name":           rec.get("name", ""),
        "region":         rec.get("name", ""),
        "city":           rec.get("name", ""),
        "latitude":       rec.get("latitude", 7.9465),
        "longitude":      rec.get("longitude", -1.0232),
        # Counts
        "total_facilities": rec.get("total_facilities", 0),
        "hospital_count":   rec.get("hospital_count", 0),
        "clinic_count":     rec.get("clinic_count", 0),
        "ngo_count":        rec.get("ngo_count", 0),
        # Workforce
        "total_doctors":   rec.get("total_doctors", 0),
        "total_beds":      rec.get("total_beds", 0),
        "doctors_per_100k":rec.get("doctors_per_100k", 0.0),
        "beds_per_100k":   rec.get("beds_per_100k", 0.0),
        # Specialty coverage
        "emergency_medicine_facilities": rec.get("emergency_medicine_facilities", 0),
        "surgery_facilities":            rec.get("surgery_facilities", 0),
        "obstetrics_facilities":         rec.get("obstetrics_facilities", 0),
        "icu_facilities":                rec.get("icu_facilities", 0),
        "infectious_disease_facilities": rec.get("infectious_disease_facilities", 0),
        "mental_health_facilities":      rec.get("mental_health_facilities", 0),
        # Gaps
        "emergency_gap_score":       rec.get("emergency_gap_score", 0.0),
        "icu_gap_score":             rec.get("icu_gap_score", 0.0),
        "surgical_access_gap_score": rec.get("surgical_access_gap_score", 0.0),
        "maternity_gap_score":       rec.get("maternity_gap_score", 0.0),
        "missing_critical_specialties": rec.get("missing_critical_specialties", []),
        "recommended_actions":          rec.get("recommended_actions", []),
        # Desert
        "medical_desert_score": mds,
        "desert_label":         rec.get("desert_label", "Unknown"),
        "risk_level":           rec.get("risk_level", "LOW"),
        # Quality
        "avg_completeness":       rec.get("avg_completeness", 0.0),
        "total_region_anomalies": rec.get("total_region_anomalies", 0),
        "rag_ready_count":        rec.get("rag_ready_count", 0),
        "avg_emergency_readiness":rec.get("avg_emergency_readiness", 0.0),
        "avg_critical_care_score":rec.get("avg_critical_care_score", 0.0),
        "avg_service_richness_score": rec.get("avg_service_richness_score", 0.0),
        "avg_infrastructure_completeness_score": rec.get("avg_infrastructure_completeness_score", 0.0),
        "avg_referral_complexity_score": rec.get("avg_referral_complexity_score", 0.0),
        "avg_healthcare_maturity_score": rec.get("avg_healthcare_maturity_score", 0.0),
        # Planning
        "medical_gaps":  [],
        "medical_needs": [],
        "idp_citations": [],
        "accepts_volunteers": False,
        "has_anomaly": False,
    }
    # Infer region-level gaps
    gaps:  List[str] = []
    needs: List[str] = []
    if rec.get("icu_gap_score", 0) > 0.5:
        gaps.append("Region-wide ICU gap (score > 0.5)")
        needs.append("ICU infrastructure investment required")
    if rec.get("emergency_gap_score", 0) > 0.5:
        gaps.append("Emergency care gap across region")
        needs.append("Emergency response units needed")
    if rec.get("surgical_access_gap_score", 0) > 0.5:
        gaps.append("Insufficient surgical access across region")
        needs.append("Surgical capacity expansion required")
    if rec.get("total_doctors", 0) == 0:
        gaps.append("No doctors recorded for this region")
        needs.append("Urgent medical workforce deployment needed")
    meta["medical_gaps"]  = gaps
    meta["medical_needs"] = needs

    return meta

# COMMAND ----------
# MAGIC %md ## 5. Embedding Utilities (BGE-large-en, 1024-dim)

# COMMAND ----------

def embed_texts(texts: List[str], batch_size: int = 8) -> np.ndarray:
    """
    Batch-embed texts via Databricks BGE-large-en endpoint.
    Returns float32 array shape (n, 1024).
    Falls back to zero-vectors on persistent failure (logged as warning).
    """
    
    if not cfg.DATABRICKS_TOKEN:
        raise ValueError("❌ No authentication token available. Cannot call embedding endpoint.")
    all_embeddings: List[List[float]] = []
    headers = {
        "Authorization": f"Bearer {cfg.DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    }
    total_batches = max(1, (len(texts) - 1) // batch_size + 1)

    for i in range(0, len(texts), batch_size):
        batch = [t[:2000] if t else " " for t in texts[i:i + batch_size]]
        batch_idx = i // batch_size + 1
        embedded = False

        for attempt in range(4):
            try:
                resp = requests.post(
                    cfg.EMBED_ENDPOINT, 
                    headers=headers,
                    json={"input": batch}, 
                    timeout=60,
                )
                
                # ✅ Better error reporting
                if resp.status_code == 403:
                    print(f"❌ 403 Forbidden: Token lacks permissions or is invalid")
                    print(f"   Endpoint: {cfg.EMBED_ENDPOINT}")
                    print(f"   Check: 1) Token validity, 2) Endpoint permissions, 3) Endpoint exists")
                    raise requests.HTTPError(f"403 Forbidden", response=resp)
                
                resp.raise_for_status()
                data = sorted(resp.json()["data"], key=lambda x: x["index"])
                all_embeddings.extend(item["embedding"] for item in data)
                embedded = True
                break
                
            except requests.HTTPError as e:
                status_code = getattr(e.response, "status_code", 0)
                if status_code == 429:
                    wait = min(2 ** attempt * 5, 60)
                    print(f"  [batch {batch_idx}/{total_batches}] Rate-limited. Waiting {wait}s …")
                    time.sleep(wait)
                elif status_code == 403:
                    # Don't retry on auth failures
                    print(f"  [batch {batch_idx}] Authentication failed. Stopping.")
                    raise
                elif attempt == 3:
                    print(f"  [batch {batch_idx}] FAILED after 4 attempts ({e})")
                    raise
                else:
                    time.sleep(2 ** attempt)
                    
        # Remove the zero-vector fallback for auth failures
        if not embedded:
            raise RuntimeError(f"Failed to embed batch {batch_idx} after all retries")

        if batch_idx % 10 == 0 or batch_idx == total_batches:
            print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} texts")
        time.sleep(cfg.RATE_LIMIT_DELAY)

    return np.array(all_embeddings, dtype=np.float32)


def embed_query(query: str) -> np.ndarray:
    """
    Embed a single query string.
    Returns L2-normalised shape (1, 1024) float32.
    """
    headers = {
        "Authorization": f"Bearer {cfg.DATABRICKS_TOKEN}",
        "Content-Type":  "application/json",
    }
    for attempt in range(3):
        try:
            resp = requests.post(
                cfg.EMBED_ENDPOINT, headers=headers,
                json={"input": [query[:2000]]}, timeout=30,
            )
            resp.raise_for_status()
            vec = np.array([resp.json()["data"][0]["embedding"]], dtype=np.float32)
            faiss.normalize_L2(vec)
            return vec
        except Exception as e:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

# COMMAND ----------
# MAGIC %md ## 6. Load Source Data

# COMMAND ----------

# ── Load IDP table ────────────────────────────────────────────────────────────
try:
    idp_pd = spark.table(cfg.IDP_TABLE).toPandas()
    print(f"[IDP]        gold_idp_enriched         : {len(idp_pd):>5} rows, {len(idp_pd.columns)} cols")
    idp_loaded = True
except Exception as e:
    print(f"[IDP]  UNAVAILABLE ({e})")
    idp_pd = pd.DataFrame()
    idp_loaded = False

# ── Load facilities table ─────────────────────────────────────────────────────
try:
    fac_pd = spark.table(cfg.FACILITIES_TABLE).toPandas()
    print(f"[Facilities] gold_facilities_enriched   : {len(fac_pd):>5} rows, {len(fac_pd.columns)} cols")
    fac_loaded = True
except Exception as e:
    print(f"[Facilities] UNAVAILABLE ({e})")
    fac_pd = pd.DataFrame()
    fac_loaded = False

# ── Load regional summary table ───────────────────────────────────────────────
try:
    reg_pd = spark.table(cfg.REGIONAL_TABLE).toPandas()
    print(f"[Regional]   gold_regional_summary_v12  : {len(reg_pd):>5} rows, {len(reg_pd.columns)} cols")
    reg_loaded = True
except Exception as e:
    print(f"[Regional]   UNAVAILABLE ({e})")
    reg_pd = pd.DataFrame()
    reg_loaded = False

if not idp_loaded and not fac_loaded:
    raise RuntimeError("Neither IDP nor Facilities table available. Cannot build index.")

# COMMAND ----------
# MAGIC %md ## 7. Load Embedding Cache

# COMMAND ----------

embedding_cache: Dict[str, List[float]] = {}
try:
    with open(cfg.LOCAL_CACHE_PATH, "rb") as f:
        embedding_cache = pickle.load(f)
    print(f"Embedding cache loaded: {len(embedding_cache):,} entries")
except FileNotFoundError:
    print("No cache found — all documents will be embedded from scratch.")
except Exception as e:
    print(f"Cache load failed ({e}) — starting fresh.")

# COMMAND ----------
# MAGIC %md ## 8. Build Document Corpus (Facility + Regional)

# COMMAND ----------

documents:  List[str]  = []
metadata:   List[Dict] = []
row_hashes: List[str]  = []
skipped:    int        = 0
source_stats: Dict[str, int] = {
    "idp": 0, "facilities": 0, "region": 0,
    "idp_doc_text": 0, "facilities_doc_text": 0,
}

# ── 8a. IDP facilities (preferred source for enriched arrays + citations) ─────
if idp_loaded:
    # Build set of IDP unique_ids to avoid duplicating with facilities table
    idp_unique_ids: set = set()
    for _, row in idp_pd.iterrows():
        rec = canonical_facility_row(row, cfg.IDP_TABLE)
        doc = build_facility_document(rec)
        if len(doc.strip()) < cfg.MIN_DOC_CHARS:
            skipped += 1
            continue
        meta = build_facility_metadata(rec)
        # Track whether we used pre-built doc_text
        if rec.get("doc_text") and len(rec.get("doc_text", "")) > cfg.MIN_DOC_CHARS:
            source_stats["idp_doc_text"] += 1
        h = hashlib.md5(doc.encode("utf-8")).hexdigest()
        documents.append(doc)
        metadata.append(meta)
        row_hashes.append(h)
        source_stats["idp"] += 1
        idp_unique_ids.add(rec.get("unique_id", ""))

print(f"IDP facilities indexed      : {source_stats['idp']:>5}"
      f"  (used pre-built doc_text for {source_stats['idp_doc_text']})")

# ── 8b. Facilities table (non-IDP rows, deduped against IDP) ──────────────────
if fac_loaded:
    for _, row in fac_pd.iterrows():
        rec = canonical_facility_row(row, cfg.FACILITIES_TABLE)
        uid = rec.get("unique_id", "")
        # Skip if already indexed from IDP
        if uid and uid in idp_unique_ids:
            continue
        doc = build_facility_document(rec)
        if len(doc.strip()) < cfg.MIN_DOC_CHARS:
            skipped += 1
            continue
        meta = build_facility_metadata(rec)
        if rec.get("doc_text") and len(rec.get("doc_text", "")) > cfg.MIN_DOC_CHARS:
            source_stats["facilities_doc_text"] += 1
        h = hashlib.md5(doc.encode("utf-8")).hexdigest()
        documents.append(doc)
        metadata.append(meta)
        row_hashes.append(h)
        source_stats["facilities"] += 1

print(f"Facilities (non-IDP) indexed: {source_stats['facilities']:>5}"
      f"  (used pre-built doc_text for {source_stats['facilities_doc_text']})")

# ── 8c. Regional summary rows ─────────────────────────────────────────────────
if reg_loaded:
    for _, row in reg_pd.iterrows():
        rec = canonical_region_row(row)
        doc = build_region_document(rec)
        if len(doc.strip()) < cfg.MIN_DOC_CHARS:
            skipped += 1
            continue
        meta = build_region_metadata(rec)
        h = hashlib.md5(doc.encode("utf-8")).hexdigest()
        documents.append(doc)
        metadata.append(meta)
        row_hashes.append(h)
        source_stats["region"] += 1

print(f"Regional summaries indexed  : {source_stats['region']:>5}")
print(f"Skipped (too short/empty)   : {skipped:>5}")
print(f"Total corpus size           : {len(documents):>5}")
print(f"\nSample facility doc ({len(documents[0])} chars):\n{documents[0][:500]}")
if source_stats["region"] > 0:
    region_start = source_stats["idp"] + source_stats["facilities"]
    print(f"\nSample region doc ({len(documents[region_start])} chars):\n{documents[region_start][:500]}")

# COMMAND ----------
# MAGIC %md ## 9. Compute Embeddings (Cache-Aware)

# COMMAND ----------

# Identify which docs need new embeddings
to_embed_indices   = [i for i, h in enumerate(row_hashes) if h not in embedding_cache]
cached_embeddings  = {
    i: np.array(embedding_cache[h], dtype=np.float32)
    for i, h in enumerate(row_hashes) if h in embedding_cache
}

print(f"Cache hits  : {len(cached_embeddings):,}")
print(f"Need embed  : {len(to_embed_indices):,}")

if to_embed_indices:
    texts_to_embed = [documents[i] for i in to_embed_indices]
    print(f"Embedding {len(texts_to_embed):,} documents in batches of {cfg.BATCH_SIZE} …")
    new_embeddings = embed_texts(texts_to_embed, batch_size=cfg.BATCH_SIZE)

    for enum_idx, doc_idx in enumerate(to_embed_indices):
        vec = new_embeddings[enum_idx].astype(np.float32)
        cached_embeddings[doc_idx] = vec
        embedding_cache[row_hashes[doc_idx]] = vec.tolist()

# Assemble final matrix in document order
all_embeddings = np.vstack([
    cached_embeddings[i].reshape(1, -1) for i in range(len(documents))
]).astype(np.float32)
print(f"\nEmbedding matrix: {all_embeddings.shape}  dtype={all_embeddings.dtype}")

# COMMAND ----------
# MAGIC %md ## 10. Build FAISS Index (IndexFlatIP — Cosine Similarity)

# COMMAND ----------

# L2-normalise → inner product == cosine similarity
faiss.normalize_L2(all_embeddings)

index = faiss.IndexFlatIP(cfg.EMBED_DIM)
index.add(all_embeddings)
print(f"FAISS IndexFlatIP: {index.ntotal:,} vectors, dim={cfg.EMBED_DIM}")

# ── Self-retrieval sanity checks ──────────────────────────────────────────────
for test_idx, label in [(0, "facility[0]"),
                        (source_stats["idp"] + source_stats["facilities"], "region[0]")]:
    if test_idx < len(documents):
        test_vec = all_embeddings[test_idx:test_idx + 1].copy()
        D, I = index.search(test_vec, 1)
        status = "✅" if I[0][0] == test_idx else f"❌ got {I[0][0]}"
        print(f"Self-retrieval [{label}]: {status}  (score={D[0][0]:.4f})")

# COMMAND ----------
# MAGIC %md ## 11. Persist Index + Metadata

# COMMAND ----------

# ── Write to local Workspace ──────────────────────────────────────────────────
faiss.write_index(index, cfg.LOCAL_INDEX_PATH)
with open(cfg.LOCAL_META_PATH, "w", encoding="utf-8") as f:
    json.dump({"documents": documents, "metadata": metadata,
               "source_stats": source_stats,
               "built_at": datetime.now(timezone.utc).isoformat(),
               "schema_version": cfg.SCHEMA_VERSION},
              f, ensure_ascii=False)
with open(cfg.LOCAL_CACHE_PATH, "wb") as f:
    pickle.dump(embedding_cache, f)

print(f"✅  Local files written to {cfg.LOCAL_DIR}")
print(f"   Index    : {cfg.LOCAL_INDEX_PATH}")
print(f"   Metadata : {cfg.LOCAL_META_PATH}")
print(f"   Cache    : {cfg.LOCAL_CACHE_PATH}")

# ── Mirror to Unity Catalog Volume ───────────────────────────────────────────
for src, dst in [
    (cfg.LOCAL_INDEX_PATH, cfg.INDEX_PATH),
    (cfg.LOCAL_META_PATH,  cfg.META_PATH),
    (cfg.LOCAL_CACHE_PATH, cfg.EMBED_CACHE_PATH),
]:
    try:
        dbutils.fs.cp(f"file:{src}", dst, recurse=False)
        print(f"  Volume: {dst}")
    except Exception as e:
        print(f"  Volume copy skipped ({type(e).__name__}): {e}")

# COMMAND ----------
# MAGIC %md ## 12. Retrieval API

# COMMAND ----------

_rag_index: Optional[faiss.IndexFlatIP] = None
_rag_store: Optional[Dict]              = None


def load_rag_index(force_reload: bool = False):
    """Load FAISS index + metadata store (singleton, cached in module scope)."""
    global _rag_index, _rag_store
    if _rag_index is None or force_reload:
        _rag_index = faiss.read_index(cfg.LOCAL_INDEX_PATH)
        with open(cfg.LOCAL_META_PATH, encoding="utf-8") as f:
            _rag_store = json.load(f)
        print(f"RAG index loaded: {_rag_index.ntotal:,} vectors")
    return _rag_index, _rag_store


def rag_search(
    query: str,
    k:     int            = 5,
    # Intent / doc-type filter
    doc_type:         Optional[str]   = None,     # "facility" | "region" | None (both)
    # Facility-specific filters
    filter_region:    Optional[str]   = None,
    filter_type:      Optional[str]   = None,     # "hospital" | "clinic" | "ngo" …
    filter_specialty: Optional[str]   = None,     # e.g. "internalMedicine"
    filter_volunteers: bool           = False,
    filter_desert_only: bool          = False,    # Critical/Severe desert only
    filter_anomalies_only: bool       = False,    # return only anomalous rows
    exclude_anomalies:  bool          = False,    # exclude anomalous rows
    filter_planning_ready: bool       = False,    # is_planning_ready == True
    filter_clinical_ready: bool       = False,    # is_clinical_ready == True
    # Quality thresholds
    min_completeness: float           = 0.0,
    min_rag_quality:  float           = 0.0,
    # Over-retrieval multiplier
    over_k: int = 8,
) -> List[Dict]:
    """
    Semantic search over the multi-corpus facility + region knowledge base.
    Strategy: over-retrieve (k × over_k), apply post-retrieval filters, return top-k.

    Returns list of {score, document, metadata, citations} dicts sorted by relevance.
    """
    idx, store = load_rag_index()
    q_vec = embed_query(query)

    retrieve_k = min(k * over_k, idx.ntotal)
    distances, indices = idx.search(q_vec, retrieve_k)

    results: List[Dict] = []
    for dist, i in zip(distances[0], indices[0]):
        if i < 0 or i >= len(store["documents"]):
            continue
        meta = store["metadata"][i]

        # ── Post-retrieval filters ────────────────────────────────────────────
        if doc_type and meta.get("doc_type", "facility") != doc_type:
            continue
        if filter_region:
            if meta.get("region", "").lower().strip() != filter_region.lower().strip():
                continue
        if filter_type:
            if meta.get("facility_type", "").lower() != filter_type.lower():
                continue
        if filter_specialty:
            specs_lower = [s.lower() for s in meta.get("specialties", [])]
            if filter_specialty.lower() not in " ".join(specs_lower):
                continue
        if filter_volunteers:
            doc_text = store["documents"][i].lower()
            if not (meta.get("accepts_volunteers") or "volunteer" in doc_text):
                continue
        if filter_desert_only:
            if meta.get("desert_label", "") not in (
                "Critical Desert", "Severe Desert", "Severe", "Critical"
            ):
                continue
        if filter_anomalies_only:
            if not meta.get("has_anomaly"):
                continue
        if exclude_anomalies:
            if meta.get("has_anomaly"):
                continue
        if filter_planning_ready:
            if not meta.get("is_planning_ready", True):
                continue
        if filter_clinical_ready:
            if not meta.get("is_clinical_ready", True):
                continue
        if meta.get("data_completeness_score", 0.0) < min_completeness:
            continue
        if meta.get("rag_quality_score", 0.0) < min_rag_quality:
            continue

        results.append({
            "score":     float(dist),
            "document":  store["documents"][i],
            "metadata":  meta,
            "citations": [c for c in meta.get("idp_citations", []) if c],
        })
        if len(results) >= k:
            break

    return results


def rag_search_facilities(query: str, k: int = 5, **kwargs) -> List[Dict]:
    """Convenience: search only facility documents."""
    return rag_search(query, k=k, doc_type="facility", **kwargs)


def rag_search_regions(query: str, k: int = 5, **kwargs) -> List[Dict]:
    """Convenience: search only regional summary documents."""
    return rag_search(query, k=k, doc_type="region", **kwargs)


def rag_search_with_citations(query: str, k: int = 5, **kwargs) -> List[Dict]:
    """
    Semantic search with IDP citation chain attached.
    Fulfils the stretch-goal citation requirement; used by synthesiser layer.
    """
    results = rag_search(query, k, **kwargs)
    for r in results:
        r["idp_run_id"]    = r["metadata"].get("idp_run_id", "")
        r["facility_id"]   = r["metadata"].get("unique_id", "")
        r["medical_gaps"]  = r["metadata"].get("medical_gaps", [])
        r["medical_needs"] = r["metadata"].get("medical_needs", [])
        r["risk_level"]    = r["metadata"].get("risk_level", "LOW")
    return results

# COMMAND ----------
# MAGIC %md ## 13. Planning & Resource Allocation System

# COMMAND ----------

_PRIORITY_ORDER = {
    "CRITICAL":    0,
    "HIGH":        1,
    "MEDIUM":      2,
    "OPPORTUNITY": 3,
    "MONITOR":     4,
    "NORMAL":      5,
}


def _facility_priority(meta: Dict) -> Tuple[str, str]:
    """
    Derive priority label and recommended action for one facility metadata dict.
    Returns (priority_str, action_str).
    """
    mds   = meta.get("medical_desert_score", 0.5)
    risk  = meta.get("risk_level", "LOW")
    needs = meta.get("medical_needs", [])
    ghost = meta.get("ghost_probability_score", 0.0)

    if ghost > 0.7:
        return "CRITICAL", "Ghost facility suspected — immediate field verification required"
    if meta.get("has_anomaly"):
        return "CRITICAL", "Investigate capability anomaly + field-verify claims"
    if risk == "HIGH" or mds > 0.6:
        action = ("Immediate deployment: " + "; ".join(needs[:3])) if needs else "Immediate support required"
        return "CRITICAL", action
    if risk == "MEDIUM" or mds > 0.35:
        action = ("Planned upgrade: " + "; ".join(needs[:2])) if needs else "Infrastructure upgrade needed"
        return "HIGH", action
    if meta.get("accepts_volunteers"):
        return "OPPORTUNITY", "Send volunteer doctors / NGO medical teams"
    if needs:
        return "MEDIUM", "Address service gaps: " + "; ".join(needs[:2])
    return "NORMAL", "Monitor — no immediate action required"


def plan_resource_allocation(
    query:        str,
    k:            int            = 10,
    focus_region: Optional[str] = None,
    doc_type:     str            = "facility",
    **kwargs,
) -> List[Dict]:
    """
    Planning system for NGO / non-technical users.
    Returns facilities or regions ranked by intervention priority.
    """
    results = rag_search(query, k=k, doc_type=doc_type,
                          filter_region=focus_region, **kwargs)
    plan: List[Dict] = []

    for r in results:
        m = r["metadata"]
        mtype = m.get("doc_type", "facility")

        if mtype == "region":
            mds = m.get("medical_desert_score", 0.5)
            rk  = m.get("risk_level", "LOW")
            actions = m.get("recommended_actions", [])
            plan.append({
                "doc_type":           "region",
                "facility_name":      m.get("name"),
                "region":             m.get("region"),
                "city":               "",
                "facility_type":      "region_summary",
                "priority":           "CRITICAL" if mds > 0.6 else "HIGH" if mds > 0.35 else "MEDIUM",
                "relevance_score":    r["score"],
                "recommended_action": "; ".join(actions[:3]) if actions else f"Regional support needed ({rk} risk)",
                "medical_gaps":       m.get("medical_gaps", []),
                "medical_needs":      m.get("medical_needs", []),
                "total_facilities":   m.get("total_facilities", 0),
                "total_doctors":      m.get("total_doctors", 0),
                "total_beds":         m.get("total_beds", 0),
                "icu_facilities":     m.get("icu_facilities", 0),
                "surgery_facilities": m.get("surgery_facilities", 0),
                "medical_desert_score": mds,
                "desert_label":       m.get("desert_label"),
                "citations":          [],
            })
        else:
            priority, action = _facility_priority(m)
            plan.append({
                "doc_type":           "facility",
                "facility_name":      m.get("name"),
                "region":             m.get("region"),
                "city":               m.get("city"),
                "facility_type":      m.get("facility_type"),
                "priority":           priority,
                "relevance_score":    r["score"],
                "recommended_action": action,
                "medical_gaps":       m.get("medical_gaps", []),
                "medical_needs":      m.get("medical_needs", []),
                "has_icu":            m.get("has_icu"),
                "has_surgery":        m.get("has_surgery"),
                "has_emergency":      m.get("has_emergency_medicine"),
                "has_obstetrics":     m.get("has_obstetrics"),
                "doctors":            m.get("doctors"),
                "beds":               m.get("capacity"),
                "medical_desert_score": m.get("medical_desert_score"),
                "desert_label":       m.get("desert_label"),
                "accepts_volunteers": m.get("accepts_volunteers"),
                "has_anomaly":        m.get("has_anomaly"),
                "ghost_probability":  m.get("ghost_probability_score"),
                "emergency_readiness":m.get("emergency_readiness_score"),
                "critical_care_score":m.get("critical_care_score"),
                "citations":          m.get("idp_citations", [])[:3],
            })

    plan.sort(key=lambda x: _PRIORITY_ORDER.get(x["priority"], 6))
    return plan


def get_volunteer_opportunities(k: int = 20) -> List[Dict]:
    """Return facilities explicitly accepting volunteers, ranked by desert severity."""
    results = rag_search(
        "volunteer doctors NGO healthcare workers",
        k=k, doc_type="facility", filter_volunteers=True
    )
    return sorted(results, key=lambda r: -r["metadata"].get("medical_desert_score", 0))


def get_cold_spots(k: int = 10) -> List[Dict]:
    """Return facilities and regions in Critical/Severe desert zones."""
    return rag_search(
        "medical desert underserved rural area no hospitals",
        k=k, filter_desert_only=True
    )


def get_anomalous_facilities(k: int = 20) -> List[Dict]:
    """Return facilities with statistical anomalies or invalid capability claims."""
    return rag_search(
        "suspicious capability claim inflated anomaly ghost facility",
        k=k, doc_type="facility", filter_anomalies_only=True
    )

# COMMAND ----------
# MAGIC %md ## 14. Map Export

# COMMAND ----------

# ── Map colour palette by desert severity ─────────────────────────────────────
_DESERT_COLOURS = {
    "Critical Desert": "#dc2626",   # red
    "Severe Desert":   "#ea580c",   # orange
    "Moderate Desert": "#ca8a04",   # amber
    "Marginal":        "#16a34a",   # green
    "Adequate":        "#2563eb",   # blue
    "Well-Served":     "#0891b2",   # cyan
}


def _desert_colour(label: str, score: float) -> str:
    if label in _DESERT_COLOURS:
        return _DESERT_COLOURS[label]
    # Fallback: score-based
    if score > 0.75:  return "#dc2626"
    if score > 0.55:  return "#ea580c"
    if score > 0.35:  return "#ca8a04"
    if score > 0.15:  return "#16a34a"
    return "#2563eb"


def export_map_data(
    query:       Optional[str] = None,
    k:           int           = 500,
    doc_types:   Optional[List[str]] = None,   # ["facility", "region"] or subset
) -> List[Dict]:
    """
    Generate GeoJSON-ready feature data for Leaflet / Plotly / Databricks map.
    If query is given: semantic search. Otherwise: return all indexed records.
    Separate facility + region layers returned in the same list, tagged by doc_type.
    """
    if doc_types is None:
        doc_types = ["facility", "region"]

    if query:
        raw_results = rag_search(query, k=k)
        items = [r["metadata"] for r in raw_results]
    else:
        _, store = load_rag_index()
        items    = store["metadata"]

    features: List[Dict] = []
    for m in items:
        if m.get("doc_type", "facility") not in doc_types:
            continue

        lat = float(m.get("latitude", 0.0))
        lon = float(m.get("longitude", 0.0))
        if lat == 0.0 and lon == 0.0:
            continue

        mds   = float(m.get("medical_desert_score", 0.5))
        dlabel= str(m.get("desert_label", "Unknown"))
        color = _desert_colour(dlabel, mds)

        if m.get("doc_type") == "region":
            features.append({
                "doc_type":     "region",
                "name":         m.get("name"),
                "lat":          lat,
                "lon":          lon,
                "region":       m.get("region"),
                "total_facilities": m.get("total_facilities"),
                "total_doctors":    m.get("total_doctors"),
                "total_beds":       m.get("total_beds"),
                "icu_facilities":   m.get("icu_facilities"),
                "surgery_facilities": m.get("surgery_facilities"),
                "emergency_medicine_facilities": m.get("emergency_medicine_facilities"),
                "missing_critical_specialties": m.get("missing_critical_specialties", []),
                "medical_desert_score": mds,
                "desert_label": dlabel,
                "risk_level":   m.get("risk_level"),
                "medical_gaps": m.get("medical_gaps", [])[:3],
                "color":        color,
                "tooltip":      f"{m.get('name','?')} Region | {dlabel} | MDS={mds:.3f}",
                "unique_id":    m.get("unique_id"),
            })
        else:
            features.append({
                "doc_type":     "facility",
                "name":         m.get("name"),
                "lat":          lat,
                "lon":          lon,
                "region":       m.get("region"),
                "city":         m.get("city"),
                "facility_type":m.get("facility_type"),
                "organization_type": m.get("organization_type"),
                "has_icu":            m.get("has_icu"),
                "has_surgery":        m.get("has_surgery"),
                "has_emergency":      m.get("has_emergency_medicine"),
                "has_obstetrics":     m.get("has_obstetrics"),
                "accepts_volunteers": m.get("accepts_volunteers"),
                "medical_desert_score": mds,
                "desert_label": dlabel,
                "has_anomaly":  m.get("has_anomaly"),
                "ghost_probability": m.get("ghost_probability_score"),
                "doctors":      m.get("doctors"),
                "beds":         m.get("capacity"),
                "completeness": m.get("data_completeness_score"),
                "rag_quality":  m.get("rag_quality_score"),
                "specialties":  m.get("specialties", [])[:5],
                "medical_gaps": m.get("medical_gaps", [])[:3],
                "color":        color,
                "tooltip":      f"{m.get('name','?')} | {m.get('facility_type','?')} | {m.get('region','?')}",
                "unique_id":    m.get("unique_id"),
            })

    return features

# COMMAND ----------
# MAGIC %md ## 15. Search Quality Tests — Must-Have Query Coverage

# COMMAND ----------

print("=" * 72)
print("RAG SEARCH — Quality Tests (Must-Have + Intent Coverage)")
print("=" * 72)

TEST_CASES = [
    # (description, query, filters, min_expected_results)
    ("1. Ashanti surgery hospitals",
     "hospitals with surgical capabilities in Ashanti",
     {"doc_type": "facility", "filter_region": "Ashanti"},              1),

    ("2. Emergency obstetric care",
     "emergency obstetric care facilities maternity Ghana",
     {"doc_type": "facility"},                                           1),

    ("3. Infectious disease / HIV / TB",
     "HIV AIDS tuberculosis TB infectious disease treatment",
     {"doc_type": "facility"},                                           1),

    ("4. Dialysis clinics",
     "clinics that perform dialysis renal",
     {"doc_type": "facility"},                                           1),

    ("5. ICU critical care",
     "ICU intensive care unit critical care ventilator",
     {"doc_type": "facility"},                                           1),

    ("6. Volunteer acceptance",
     "facilities accepting volunteer doctors NGOs medical teams",
     {"doc_type": "facility", "filter_volunteers": True},               1),

    ("7. Anomaly / ghost facilities",
     "suspicious capability inflation anomaly ghost facility",
     {"doc_type": "facility", "filter_anomalies_only": True},           0),  # may be 0

    ("8. Greater Accra malaria treatment",
     "malaria treatment hospitals near Accra",
     {"doc_type": "facility", "filter_region": "Greater Accra"},        1),

    ("9. Medical desert cold spots (regions)",
     "regions with critical medical desert severe underserved cold spot",
     {"doc_type": "region"},                                             1),

    ("10. Regional surgery coverage",
     "which regions lack surgical access surgery coverage gap",
     {"doc_type": "region"},                                             1),

    ("11. Upper West planning",
     "most at-risk facilities needing emergency support",
     {"doc_type": "facility", "filter_region": "Upper West"},           0),

    ("12. Dental / ophthalmology specialty",
     "dental services ophthalmology eye care specialist",
     {"doc_type": "facility"},                                           1),

    ("13. Teaching / referral hospitals",
     "teaching hospital referral centre tertiary care complex",
     {"doc_type": "facility"},                                           1),

    ("14. Mental health services",
     "mental health psychiatry services psychiatric hospital",
     {"doc_type": "facility"},                                           1),

    ("15. NGO healthcare",
     "NGO non-governmental healthcare facilities serving Ghana",
     {"doc_type": "facility"},                                           1),
]

PASS = 0
TOTAL_REQUIRED = sum(1 for *_, min_r in TEST_CASES if min_r > 0)

for desc, query, filters, min_results in TEST_CASES:
    results = rag_search(query, k=3, **filters)
    met = len(results) >= min_results
    if met or min_results == 0:
        status = "✅"
        if len(results) >= max(min_results, 1):
            PASS += 1
    else:
        status = f"⚠️  ({len(results)}/min {min_results})"

    print(f"\n[{status}] {desc}")
    print(f"  Query: {query!r}")
    for j, r in enumerate(results[:2]):
        m   = r["metadata"]
        afl = "⚠️" if m.get("has_anomaly") else "✅"
        dtype = m.get("doc_type", "facility")
        if dtype == "region":
            print(f"  [{j+1}] 🗺  score={r['score']:.4f}  REGION: {m['region']}  "
                  f"desert={m.get('desert_label','?')}")
        else:
            print(f"  [{j+1}] {afl}  score={r['score']:.4f}  "
                  f"{m.get('name','?')!r:<40} "
                  f"{m.get('facility_type','?'):<10}  "
                  f"{m.get('city','?')}, {m.get('region','?')}")

print(f"\n{'='*72}")
print(f"Tests PASSED: {PASS} / {len(TEST_CASES)}  (required ≥1 result: {PASS}/{TOTAL_REQUIRED})")

# COMMAND ----------
# MAGIC %md ## 16. Citation-Backed Retrieval (IDP Stretch Goal)

# COMMAND ----------

print("=" * 72)
print("CITATION-BACKED RETRIEVAL  (IDP Stretch Goal)")
print("=" * 72)

citation_tests = [
    "surgical facilities with trauma and emergency capabilities",
    "facilities claiming ICU without supporting equipment",
    "hospitals offering obstetric C-section caesarean delivery",
]

for query in citation_tests:
    print(f"\n📋 Query: {query!r}")
    results = rag_search_with_citations(query, k=3, doc_type="facility")
    for r in results:
        m = r["metadata"]
        print(f"  📍 {m.get('name','?')} ({m.get('region','?')}, {m.get('city','?')})")
        print(f"     Score     : {r['score']:.4f}  |  Desert: {m.get('desert_label','?')}")
        print(f"     Risk      : {r.get('risk_level','?')}  |  IDP run: {r.get('idp_run_id','N/A')[:16]}")
        if r["citations"]:
            print(f"     Citations ({len(r['citations'])}):")
            for c in r["citations"][:2]:
                print(f"       → {str(c)[:120]}")
        if r.get("medical_gaps"):
            print(f"     Gaps : {'; '.join(r['medical_gaps'][:2])}")

# COMMAND ----------
# MAGIC %md ## 17. Planning System Tests

# COMMAND ----------

print("=" * 72)
print("PLANNING SYSTEM OUTPUT")
print("=" * 72)

plan_tests = [
    ("hospitals needing surgery support", "facility", None),
    ("most at-risk regional gaps — cold spots", "region", None),
    ("clinics with no emergency services Volta", "facility", "Volta"),
    ("facilities needing immediate intervention Upper West", "facility", "Upper West"),
]

for query, dtype, region in plan_tests:
    print(f"\n📊 Plan: {query!r}  [doc_type={dtype}, region={region or 'All'}]")
    plan = plan_resource_allocation(query, k=5, focus_region=region, doc_type=dtype)
    for p in plan[:3]:
        dstr = "🗺 REGION" if p.get("doc_type") == "region" else "🏥 FACILITY"
        print(f"  {dstr}: {p['facility_name']} ({p.get('region','?')})")
        print(f"     Priority  : {p['priority']}")
        print(f"     Action    : {p['recommended_action']}")
        print(f"     MDS score : {p['medical_desert_score']:.3f}  [{p['desert_label']}]")
        if p.get("medical_gaps"):
            print(f"     Gaps      : {'; '.join(p['medical_gaps'][:2])}")

# COMMAND ----------
# MAGIC %md ## 18. Volunteer Opportunities & Cold Spots

# COMMAND ----------

print("=" * 72)
print("VOLUNTEER OPPORTUNITIES")
print("=" * 72)
vols = get_volunteer_opportunities(k=10)
for r in vols[:5]:
    m = r["metadata"]
    print(f"  🤝  {m.get('name','?')!r:<40}  {m.get('region','?'):<20}  "
          f"MDS={m.get('medical_desert_score',0):.3f}  [{m.get('desert_label','?')}]")

print(f"\n{'='*72}")
print("COLD SPOTS (Severe / Critical Desert)")
print("=" * 72)
cold = get_cold_spots(k=10)
for r in cold[:5]:
    m = r["metadata"]
    dtype = m.get("doc_type", "facility")
    tag   = "🗺 REGION" if dtype == "region" else "🏥"
    print(f"  {tag}  {m.get('name','?')!r:<40}  {m.get('region','?'):<20}  "
          f"MDS={m.get('medical_desert_score',0):.3f}  [{m.get('desert_label','?')}]")

# COMMAND ----------
# MAGIC %md ## 19. Map Data Export Test

# COMMAND ----------

print("=" * 72)
print("MAP DATA EXPORT")
print("=" * 72)

map_all      = export_map_data(doc_types=["facility", "region"])
map_facility = export_map_data(doc_types=["facility"])
map_region   = export_map_data(doc_types=["region"])
map_query    = export_map_data("facilities with ICU or surgery Northern Ghana", k=50)

print(f"All features (facility+region) : {len(map_all):,}")
print(f"Facility features only         : {len(map_facility):,}")
print(f"Region features only           : {len(map_region):,}")
print(f"ICU/surgery Northern query     : {len(map_query):,}")

label_dist  = Counter(f["desert_label"] for f in map_facility)
region_dist = Counter(f["region"]       for f in map_facility)
print("\nFacility desert label distribution:")
for label, count in sorted(label_dist.items(), key=lambda x: -x[1]):
    print(f"  {label:<25} {count:>4}")
print("\nTop 6 regions by facility count:")
for region, count in region_dist.most_common(6):
    print(f"  {region:<30} {count:>4}")

print("\nRegion-level features:")
for f in map_region:
    print(f"  🗺  {f['name']:<20}  MDS={f['medical_desert_score']:.3f}  "
          f"[{f['desert_label']}]  facilities={f['total_facilities']}")

# COMMAND ----------
# MAGIC %md ## 20. MLflow Logging + Index Statistics

# COMMAND ----------

try:
    mlflow.set_experiment(cfg.MLFLOW_EXP)
    with mlflow.start_run(run_name="05_rag_build_index") as run:
        # Parameters
        mlflow.log_param("schema_version",   cfg.SCHEMA_VERSION)
        mlflow.log_param("embed_model",      cfg.EMBED_MODEL)
        mlflow.log_param("embed_dim",        cfg.EMBED_DIM)
        mlflow.log_param("index_type",       "IndexFlatIP")
        mlflow.log_param("source_idp",       cfg.IDP_TABLE)
        mlflow.log_param("source_facilities",cfg.FACILITIES_TABLE)
        mlflow.log_param("source_regional",  cfg.REGIONAL_TABLE)

        # Metrics
        mlflow.log_metric("total_documents",        len(documents))
        mlflow.log_metric("idp_docs",               source_stats["idp"])
        mlflow.log_metric("facility_docs",          source_stats["facilities"])
        mlflow.log_metric("region_docs",            source_stats["region"])
        mlflow.log_metric("cache_hits",             len(embedding_cache) - len(to_embed_indices))
        mlflow.log_metric("newly_embedded",         len(to_embed_indices))
        mlflow.log_metric("skipped_rows",           skipped)
        mlflow.log_metric("test_cases_passed",      PASS)

        # Content distribution
        all_meta = [m for m in metadata if m.get("doc_type") == "facility"]
        anomaly_ct  = sum(1 for m in all_meta if m.get("has_anomaly"))
        vol_ct      = sum(1 for m in all_meta if m.get("accepts_volunteers"))
        desert_ct   = sum(1 for m in all_meta
                          if m.get("desert_label") in ("Critical Desert", "Severe Desert",
                                                       "Severe", "Critical"))
        ghost_ct    = sum(1 for m in all_meta if m.get("ghost_probability_score", 0) > 0.5)
        mlflow.log_metric("facilities_with_anomaly",      anomaly_ct)
        mlflow.log_metric("facilities_accept_volunteers", vol_ct)
        mlflow.log_metric("high_desert_facilities",       desert_ct)
        mlflow.log_metric("potential_ghost_facilities",   ghost_ct)

        # Distribution artefact
        region_counts = Counter(m.get("region", "Unknown") for m in all_meta)
        type_counts   = Counter(m.get("facility_type", "Unknown") for m in all_meta)
        desert_counts = Counter(m.get("desert_label", "Unknown") for m in all_meta)
        mlflow.log_dict({
            "region_distribution":  dict(region_counts),
            "type_distribution":    dict(type_counts),
            "desert_distribution":  dict(desert_counts),
            "source_stats":         source_stats,
        }, "index_distribution.json")

        print(f"MLflow run logged: {run.info.run_id}")
except Exception as e:
    print(f"MLflow skipped: {e}")

# ── Final summary ─────────────────────────────────────────────────────────────
idx_l, store_l = load_rag_index(force_reload=True)
facility_meta  = [m for m in store_l["metadata"] if m.get("doc_type") == "facility"]
region_meta    = [m for m in store_l["metadata"] if m.get("doc_type") == "region"]

print(f"\n{'='*72}")
print(f"✅  RAG INDEX v12 built and validated")
print(f"   Total vectors    : {idx_l.ntotal:,}  (FAISS IndexFlatIP, dim={cfg.EMBED_DIM})")
print(f"   Facility docs    : {len(facility_meta):,}  "
      f"(IDP={source_stats['idp']}, Facilities={source_stats['facilities']})")
print(f"   Region docs      : {len(region_meta):,}")
print(f"   Embed model      : {cfg.EMBED_MODEL}")
print(f"   Index path       : {cfg.LOCAL_INDEX_PATH}")
print(f"   Volume path      : {cfg.INDEX_PATH}")
print(f"   Schema version   : {cfg.SCHEMA_VERSION}")

region_counts = Counter(m.get("region", "Unknown") for m in facility_meta)
type_counts   = Counter(m.get("facility_type", "Unknown") for m in facility_meta)
desert_counts = Counter(m.get("desert_label", "Unknown") for m in facility_meta)

print("\nFacilities by region:")
for r, c in sorted(region_counts.items(), key=lambda x: -x[1]):
    print(f"  {r:<30} {c:>4}")
print("\nFacilities by type:")
for t, c in sorted(type_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {t:<25} {c:>4}")
print("\nFacilities by desert label:")
for d, c in sorted(desert_counts.items(), key=lambda x: -x[1]):
    print(f"  {d:<25} {c:>4}")
    
    
"""Output
  FAISS   : 1.8.0
NumPy   : 1.26.4
Run at  : 2026-05-08T21:33:30.587224+00:00
✅ Databricks Host: dbc-147ceb0b-b41d.cloud.databricks.com
✅ Token available: True
✅ Embedding endpoint: https://dbc-147ceb0b-b41d.cloud.databricks.com/serving-endpoints/databricks-bge-large-en/invocations
✅ Embedding endpoint working! Vector shape: (1, 1024)
IDP table      : virtue_foundation.ghana.gold_idp_enriched
Facilities tbl : virtue_foundation.ghana.gold_facilities_enriched
Regional tbl   : virtue_foundation.ghana.gold_regional_summary
Local RAG dir  : /Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/rag
  ✅  ensure_list('["a","b"]') → ['a', 'b']
  ✅  ensure_list('[]') → []
  ✅  ensure_list('None') → []
  ✅  ensure_list('[]') → []
  ✅  ensure_list("['a', 'b']") → ['a', 'b']
  ✅  ensure_list('["internalMedicine"]') → ['internalMedicine']
  ✅  ensure_list('null') → []
  ✅  ensure_list('single') → ['single']
  ✅  ensure_list('nan') → []
[IDP]        gold_idp_enriched         :    50 rows, 181 cols
[Facilities] gold_facilities_enriched   :   955 rows, 170 cols
[Regional]   gold_regional_summary_v12  :    17 rows, 66 cols
Embedding cache loaded: 3,703 entries
IDP facilities indexed      :    50  (used pre-built doc_text for 50)
Facilities (non-IDP) indexed:   905  (used pre-built doc_text for 905)
Regional summaries indexed  :    17
Skipped (too short/empty)   :     0
Total corpus size           :   972

Sample facility doc (1417 chars):
FACILITY: Accra Psychiatric Hospital
LOCATION: Accra, Greater Accra
TYPE: hospital
SPECIALTIES: psychiatry, emergencyMedicine, pathology, physicalMedicineAndRehabilitation, internalMedicine
PROCEDURES: Laboratory, Laboratory Services, Occupational Therapy, Electro Convulsive Therapy, Emergency Care
EQUIPMENT: Laboratory Equipment, Microscope, Centrifuge
DESCRIPTION: OPD services; 24hr Emergency care; in-patients services; Clinical Psychology services; Electro Convulsive Therapy; Laboratory Servi

Sample region doc (804 chars):
Region: Western, Ghana. Medical Desert Status: Marginal (score=0.568). Healthcare infrastructure: 75 total facilities, 28 hospitals, 42 clinics, 1 NGOs, 18 government facilities. Workforce: 0 doctors total (0.00 per 100k population), 193 beds total (8.12 per 100k). Specialty coverage: 7 with emergency medicine, 3 with surgery, 18 with obstetrics, 4 with pediatrics, 2 with ICU, 1 with infectious disease, 5 with radiology, 3 with mental health. Healthcare gaps: emergency access gap=0.75, ICU gap=0
Cache hits  : 0
Need embed  : 972
Embedding 972 documents in batches of 8 …
  Embedded 80/972 texts
  Embedded 160/972 texts
  Embedded 240/972 texts
"""
